# 3 - IP Merge IFSAR Tiles by HUC8 Extent


The intended use of this tool is to query a directory containing raw IFSAR tiles using the extents of National Hydrography Dataset (NHD) HUC 8 polygons. (NHD data can be downloaded here: https://www.usgs.gov/national-hydrography/access-national-hydrography-products.) Tiles falling within or along the HUC8 polygon boundaries will be mosaicked together, then clipped to the HUC8 polygon extent.

## Required Software:

- The code contained in this notebook is designed to be run within an ESRI ArcPro project. The script output is written to the default project geodatabase and therefore the user must open and run the notebook .IPYNB file within an ArcPro project.
- If any of the geoprocessing steps require an advanced license or any specific extensions, the script will check for these conditions before running.


## Required Inputs:

- A directory containing ONLY raw IFSAR tiles as geotiffs. (The script lists all geotiffs in the given directory, so this directory should be used exclusively for raw IFSAR data.) For the Yukon-Kuskokwim IP project, these 5m IFSAR Digital Surface Model (DSM) tiles were downloaded from the USGS National Map online application for a download option (https://apps.nationalmap.gov/downloader/). See the Data.gov catalog entry for metadata (https://catalog.data.gov/dataset/5-meter-alaska-digital-elevation-models-dems-usgs-national-map-3dep-downloadable-data-collectio).

- A geodatabase containing NHD HUC8 polygons which overlay the synthetic stream reaches, which must have "HUC8" suffix in the feature class name.

- A file system output directory to store the IFSAR mosaics.


## Geoprocessing Output:


- A collection of mosaiced IFSAR DEMs clipped to HUC8 polygon extents, saved in the user-defined output directory with a suffix indicating the unique HUC 8 ID number, followed by "clip".


## Processing Steps:

1. Search through the user-defined directory containing raw IFSAR tiles, listing all ".tif" files.
2. Using the "HUC8" suffix, find the HUC8 polygon feature class in the user-defined geodatabase.
3. Create a empty dictionary, and populate each key with a unique HUC8 polygon ID. Populate dictionary values with a list of any IFSAR tiles that are contained by or overlap with the extents of each HUC8 polygon.
4. Looping through each key in the dictionary, remove any keys that may have an empty list. (This would indicate missing or nonexistent IFSAR tiles for a given HUC8 polygon.)
5. Loop through each remaining key in the dictionary, and mosaic the list of IFSAR tiles to a new raster saved in the user-defined output directory. Save with the dictionary key as the filename.
6. For each new mosaicked raster, clip using the HUC8 polygon used to define the mosaic boundaries.
7. Finally, delete the original mosaicked IFSAR raster, leaving only the clipped mosaics in the user-defined output directory.

### Code starts here:

#### Setup

Import modules. 

User provides filepaths to the geodatabase, IFSAR directory, and output directory. Set environment to the user-provided geodatabase. Additionally, prevent the addition of intermediary outputs to the ArcPro project map.

(These inputs are provided directly as text when running the code block from the notebook. Use the hashed out code block below if saving the notebook as a .PY file and creating a tool for use in the ArcPro GUI. In that case, the inputs will be provided as text parameters in "point-and-click" fashion when setting up the tool in the ArcPro GUI.)

In [72]:
import arcpy
import os
from datetime import datetime

In [73]:
work = "F:/GIS/IP/Yukon_Tanana.gdb"

ifsar_dir = "F:/GIS/IP_DATA/IFSAR"

outdir = "F:/GIS/IP_DATA/IFSAR_MOSAICS"

arcpy.env.workspace = work

arcpy.env.addOutputsToMap = False

In [3]:
#work = arcpy.GetParameterAsText(0)
#ifsar_dir = arcpy.GetParameterAsText(1)
#outdir = arcpy.GetParameterAsText(2)

#arcpy.env.workspace = work

#arcpy.env.addOutputsToMap = False

Search through the user-defined directory containing raw IFSAR tiles, listing all ".tif" files. **There should be no other TIF files in this directory or any subdirectories!**  Provide a message indicating the number of files found.

In [74]:
ifsar_rasters = []

print(str("Looking for all IFSAR rasters in " + ifsar_dir + "..."))
print(3 * "\n")
arcpy.AddMessage(str("Looking for all IFSAR rasters in " 
                     + ifsar_dir + "..."))
print(arcpy.GetMessages())

# find rasters ending with ".....tif" and append to the ifsar rasters list
walk = arcpy.da.Walk(ifsar_dir, topdown=True, datatype="RasterDataset")
for dirpath, dirnames, filenames in walk:
    for filename in filenames:
        if filename.endswith(".tif"):
            ifsar_rasters.append(os.path.join(dirpath, filename))

            
ifsar_count = len(ifsar_rasters)

# if there are no IFSAR rasters, provide an error message. 
#If there are IFSAR rasters, provide a count.
if ifsar_count == 0:
    print("User directory has no IFSAR rasters.")
    print(3 * "\n")
    arcpy.AddError("User directory has no IFSAR rasters.")
    print(arcpy.GetMessages())
    raise arcpy.ExecuteError
else:
    print("User directory has {0} IFSAR rasters:".format(ifsar_count))
    print(2 * "\n")
    print(*ifsar_rasters, sep = "\n")
    print(3 * "\n")
    arcpy.AddMessage("""User directory has {0} IFSAR rasters.""".format(ifsar_count))
    print(arcpy.GetMessages())

Looking for all IFSAR rasters in F:/GIS/IP_DATA/IFSAR...




Start Time: Wednesday, August 3, 2022 9:36:51 AM
Succeeded at Wednesday, August 3, 2022 9:36:52 AM (Elapsed Time: 0.73 seconds)
User directory has 3389 IFSAR rasters:



F:\GIS\IP_DATA\IFSAR\USGS_NED_DSM_AK_NRCS_L2_C259_2012_TIFF_2015\DSM_N6100W14215P.tif
F:\GIS\IP_DATA\IFSAR\USGS_NED_DSM_AK_NRCS_L2_C259_2012_TIFF_2015\DSM_N6100W14230P.tif
F:\GIS\IP_DATA\IFSAR\USGS_NED_DSM_AK_NRCS_L2_C259_2012_TIFF_2015\DSM_N6100W14245P.tif
F:\GIS\IP_DATA\IFSAR\USGS_NED_DSM_AK_NRCS_L2_C259_2012_TIFF_2015\DSM_N6100W14300P.tif
F:\GIS\IP_DATA\IFSAR\USGS_NED_DSM_AK_NRCS_L2_C259_2012_TIFF_2015\DSM_N6115W14215P.tif
F:\GIS\IP_DATA\IFSAR\USGS_NED_DSM_AK_NRCS_L2_C259_2012_TIFF_2015\DSM_N6115W14230P.tif
F:\GIS\IP_DATA\IFSAR\USGS_NED_DSM_AK_NRCS_L2_C259_2012_TIFF_2015\DSM_N6115W14245P.tif
F:\GIS\IP_DATA\IFSAR\USGS_NED_DSM_AK_NRCS_L2_C259_2012_TIFF_2015\DSM_N6115W14300P.tif
F:\GIS\IP_DATA\IFSAR\USGS_NED_DSM_AK_NRCS_L2_C259_2012_TIFF_2015\DSM_N6130W14215P

List all feature classes in the user-defined geodatabase that have a "HUC8" suffix but do not have a "reach" prefix. This should narrow down the selection to the NHD HUC8 polygon feature class output from the "IP 1 Subset HUC8 Polygons by NetMap Extent" script found in the IP project toolbox.

In [75]:
featureclasses = arcpy.ListFeatureClasses()

for fc in featureclasses:
    if ((fc.endswith("HUC8") == True) and (fc.startswith("reach") == False)):
        huc8_poly = fc
    else:
        pass

Check to make sure there is only one HUC8 polygon feature class.

In [76]:
print(huc8_poly)

RCA_1908311_HUC8


In [77]:
huc8_poly = 'TananaHUC8'
print(huc8_poly)

TananaHUC8


Create an empty dictionary. For each HUC8 polygon feature, determine its extent. Then loop through each IFSAR raster and compare its extent to the HUC8 polygon extent. If the HUC8 polygon contains or overlaps the IFSAR extent, add the IFSAR tiles to a new list.  

Once all IFSAR tiles are evaluated, create a new key in the dictionary named after the HUC8 polygon ID with a "DEM_5m_" prefix, and assign the IFSAR list as the value returned by that dictionary key.  

Repeat for all HUC8 polygons.

In [78]:
HUC8_IFSAR_Dictionary = {}

with arcpy.da.SearchCursor (huc8_poly, ['SHAPE@', 'huc8']) as cursor:
    for shp in cursor:
        
        h_ext = shp[0].extent

        i_list = []

        for i in ifsar_rasters:
            i_desc = arcpy.Describe(i)
            i_ext = i_desc.extent

            if h_ext.contains(i_ext) == True:
                i_list.append(i)
            elif h_ext.overlaps(i_ext) == True:
                i_list.append(i)
            else:
                pass

        name = str("DEM_5m_" + str(shp[1]))
        HUC8_IFSAR_Dictionary[name] = i_list

Check the list of dictionary keys to ensure that there is only one key per HUC8 polygon.

In [79]:
huc8_list = list(HUC8_IFSAR_Dictionary.keys())
huc8_list

['DEM_5m_19080301', 'DEM_5m_19080302', 'DEM_5m_19080305', 'DEM_5m_19080309', 'DEM_5m_19080308', 'DEM_5m_19080310', 'DEM_5m_19080304', 'DEM_5m_19080306', 'DEM_5m_19080311', 'DEM_5m_19080307', 'DEM_5m_19080303']

Run a check to make sure that there are no keys returning an empty list. This could be the case for any HUC8 polygons in the dataset that do not overlap or contain any IFSAR tiles in the directory. If any empty lists are encountered, removed the key from the dictionary to prevent mosaicking an empty list in the next step.

In [80]:
for h in huc8_list:
    if HUC8_IFSAR_Dictionary[h] == '':
        huc8_list.remove(h)
    else:
        pass

For each key in the dictionary, return the list of IFSAR tiles and mosaic them together. Save the new mosaicked raster in the user-defined output directory using the dictionary key as the filename.

In [81]:
mosaics = []

print("Mosaicking IFSAR rasters together....")
print(3 * "\n")
arcpy.AddMessage("Mosaicking IFSAR rasters together....")
print(arcpy.GetMessages())

for huc8 in huc8_list:
    
    name = str(huc8 + ".tif")
    
    fullname = str(outdir + "/" + name)
    mosaics.append(fullname)
    
    arcpy.MosaicToNewRaster_management(HUC8_IFSAR_Dictionary[huc8], outdir, name, "", 
                                   "32_BIT_FLOAT", "", "1", "MAXIMUM", "")


Mosaicking IFSAR rasters together....




Start Time: Wednesday, August 3, 2022 9:36:51 AM
Succeeded at Wednesday, August 3, 2022 9:36:52 AM (Elapsed Time: 0.73 seconds)


Before clipping the mosaics, run another test to see if the number of dictionary keys matches the number of features in the HUC8 polygon feature class. (Again, this could be the case for any HUC8 polygons in the dataset that do not overlap or contain any IFSAR tiles in the directory.) Since we are going to zip the polygon features to the mosaics, that case could cause a mismatch during the clip process and result in errors.

If there is a mismatch, provide a message indicating which HUC8 polygons should be removed. Use this check to decide whether or not the clipping operation will run.

In [82]:
if int(arcpy.GetCount_management(huc8_poly).getOutput(0)) == len(mosaics):
    print("There is one mosaic for each HUC8 polygon feature...OK to proceed to the clipping step...")
    
    err = "no"
    
else:
    
    err = "yes"
    
    print("There is a mismatch between IFSAR mosaics and HUC8 polygon features... clipping step aborted!")
    
    h_list = []
    present = []
    
    with arcpy.da.SearchCursor (huc8_poly, ['huc8']) as cursor:
        for h in cursor:
            
            h_list.append(h[0])
            
            for m in mosaics:
                if str(h[0]) in m == True:
                    present.append(h[0])
                else:
                    pass
    
    for p in present:
        if p in h_list:
            h_list.remove(p)
    
    if len(h_list) > 0:
        print("Please remove the following HUC8 polygons, or add IFSAR tiles within these polygons, then try running this script again:")
        print(h_list)
        print("Alternatively, clip the mosaicked DEMs manually. The unclipped mosaics can be found in the user-defined output directory.")
            

There is one mosaic for each HUC8 polygon feature...OK to proceed to the clipping step...


If there were no mismatch errors, clip the IFSAR mosaics using the HUC8 polygons. The list of mosaics and the list of polygons provided by the search cursor should be in the same order, so these lists can be zipped together and iterated without errors.

In [83]:
if err == "no":

    with arcpy.da.SearchCursor (huc8_poly, ['SHAPE@']) as cursor:
        for shp, m in zip(cursor, mosaics):

            n = m.split(".tif")
            fullout = str(n[0] + "_clip.tif")

            arcpy.management.Clip(m, "", fullout, shp[0], "", "ClippingGeometry", "NO_MAINTAIN_EXTENT")
            
if err == "yes":
    print("Clipping skipped...")

Delete intermediary mosaic rasters, leaving only the clipped rasters in the user-defined directory. Alternatively, if there was a mismatch error, do not delete the intermediary mosaics....this will allow the user to manually clip the mosaics, if desired.

In [84]:
if err == "no":

    for d in mosaics:
        try:
            arcpy.management.Delete(d)
        except:
            desc = arcpy.Describe(d)
            arcpy.management.Delete(desc.path)
            
if err == "yes":
    print("Intermediary mosaics not deleted...check user-defined output directory for unclipped mosaics.")
        
